# Day 6 · Lab 1 — Requirements Extraction Pipeline

## What you'll build

1. Load a realistic **product planning meeting transcript**
2. **Chunk** it by speaker turns
3. Build a **Pydantic Requirement schema** with source-utterance traceability
4. Use **`llm.with_structured_output()`** for typed extraction
5. Run the staged pipeline on each chunk
6. **Verify traceability** — every requirement's source_utterance must exist in the transcript
7. **Detect duplicates** and handle a withdrawal (Marco revises 50% → 40%)

## Prerequisites

- Track 3.A + Day 5 completed
- Same sandbox: `~/agentic-lab/.env` with `OPENROUTER_API_KEY`

## Step 1 — Environment

In [ ]:
import os, sys, subprocess
from pathlib import Path

for pkg in ["python-dotenv", "langchain-openai"]:
    try:
        __import__(pkg.replace("-", "_").split("[")[0])
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pkg])

from dotenv import load_dotenv
load_dotenv(Path("~/agentic-lab/.env").expanduser(), override=False)
for k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY"):
    if os.environ.get(k) == "":
        del os.environ[k]

assert os.environ.get("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY missing"
os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
print("✓ Environment ready")

## Step 2 — Load the sample transcript

In [ ]:
TRANSCRIPT = '''Sarah (Product Manager): OK so we're here to nail down what Phase 2 of the customer portal looks like. Marco, can you kick us off?

Marco (Business Sponsor): Sure. The main driver is reducing support ticket volume. Our team is drowning — 4,000 tickets a month, most of them are 'where is my order' type questions. If we can cut that by half through self-service, that's a $200K annual savings.

Sarah (Product Manager): Great, that's a clear business goal. So the ask is a self-service order tracking feature.

Priya (Tech Lead): Before we go there — what's our latency target for order lookup? Right now the API takes about 800ms. That's not going to feel snappy in a portal.

Marco (Business Sponsor): We need it under 200ms at the 95th percentile. Anything slower and customers just call anyway.

Priya (Tech Lead): OK that's non-trivial. We'll need a read replica plus caching. Doable.

Sarah (Product Manager): Let me capture the functional side. Customers must be able to search for their orders by order number, date range, and status. And filter by any combination.

Marco (Business Sponsor): Also — they should see estimated delivery date, not just shipped/not-shipped.

Priya (Tech Lead): That means we need to integrate with the carrier APIs. FedEx and UPS at minimum. Might be an issue with rate limits on their end.

Sarah (Product Manager): Add it as a requirement, we'll deal with rate limits later. Anything else on functional?

Devon (QA Lead): I want to make sure we have clear acceptance criteria. What defines 'done' for search? Given a valid order number, when the customer searches, then results appear in under 2 seconds — is that reasonable?

Priya (Tech Lead): Yes. And given a partial or invalid input, when they submit, then a helpful error message with suggestions.

Marco (Business Sponsor): One more business point — this has to launch by Q2. Anything after that misses the peak return season.

Sarah (Product Manager): Noted. Priya, from a security standpoint — do we need MFA on portal access?

Priya (Tech Lead): For this scope, existing SSO is enough. If we add refund initiation in Phase 3, then yes MFA.

Sarah (Product Manager): Devon, one more acceptance criterion — given a customer looks up an order that isn't theirs, when they submit, then no data leaks and access is logged.

Devon (QA Lead): Absolutely, that's a hard requirement for security review.

Marco (Business Sponsor): Actually, let me revise something. I said 50% ticket reduction earlier — my team pointed out the real target from finance is 40% reduction, not 50%. Let's go with 40%.

Sarah (Product Manager): Updated. OK I think we have enough to draft the BRD. Priya can you send me your carrier integration doc? And Devon, we'll circle back on the full acceptance criteria list.'''

print(f"Transcript length: {len(TRANSCRIPT)} chars, ~{len(TRANSCRIPT.split())} words")
print("\nFirst 300 chars:")
print(TRANSCRIPT[:300])

## Step 3 — Chunk by speaker turns

Each turn is a coherent unit. Overlap by 1 turn to preserve context across boundaries.

In [ ]:
import re

def chunk_by_speaker_turns(transcript: str, overlap: int = 1) -> list[str]:
    # Split on speaker name pattern: "Name (Role): text"
    turns = re.split(r'\n(?=\w+ \([^)]+\):)', transcript.strip())
    turns = [t.strip() for t in turns if t.strip()]
    
    # Build overlapping chunks: 3 turns per chunk, 1 turn overlap
    chunks = []
    i = 0
    while i < len(turns):
        chunk = turns[i:i+3]
        chunks.append("\n\n".join(chunk))
        i += 3 - overlap
    return chunks


chunks = chunk_by_speaker_turns(TRANSCRIPT)
print(f"Transcript split into {len(chunks)} chunks:\n")
for i, ch in enumerate(chunks):
    print(f"--- Chunk {i+1} ---")
    print(ch[:250] + ("..." if len(ch) > 250 else ""))
    print()

## Step 4 — Pydantic Requirement schema

The extraction contract. Every field matters — no free text allowed.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class Requirement(BaseModel):
    id: str = Field(description="e.g. REQ-001")
    type: Literal["functional", "non_functional", "business", "acceptance"]
    statement: str = Field(description="The requirement itself, concise")
    priority: Literal["must", "should", "could", "wont"] = "should"
    source_utterance: str = Field(description="Verbatim quote from the transcript")
    source_speaker: str = Field(description="Speaker name and role")
    confidence: float = Field(ge=0.0, le=1.0, description="How confident the extractor is")
    is_withdrawal: bool = Field(default=False, description="True if speaker retracted this")


class Extraction(BaseModel):
    requirements: list[Requirement]


print("✓ Schema defined")
print(f"  Requirement fields: {list(Requirement.model_fields.keys())}")

## Step 5 — LLM extraction with structured output

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)

structured_llm = llm.with_structured_output(Extraction)


def extract_from_chunk(chunk: str, chunk_id: int) -> list[Requirement]:
    prompt = f"""You are extracting business/product requirements from a meeting transcript chunk.

For each requirement:
- Type: functional (what system does) / non_functional (SLA, security, perf) / business (KPI, goal, constraint) / acceptance (Given/When/Then, testable)
- Copy source_utterance VERBATIM from the transcript. Do not paraphrase.
- Detect withdrawals: if a speaker says "actually, scratch that" or "let me revise", set is_withdrawal=True
- Skip: chatter, questions, undecided items, opinions without commitment
- Priority: 'must' if speaker used strong language ("has to", "must", "hard requirement"); 'should' by default

Chunk (chunk_id={chunk_id}):
""""
{chunk}
""""

Assign incrementing IDs: REQ-{{chunk_id}}01, REQ-{{chunk_id}}02, etc.
Confidence: 1.0 for explicit statement; 0.7 for implied; 0.4 for speculative."""

    result = structured_llm.invoke(prompt)
    return result.requirements


# Test extraction on chunk 1
sample_reqs = extract_from_chunk(chunks[0], 1)
print(f"Extracted {len(sample_reqs)} requirement(s) from chunk 1:")
for r in sample_reqs:
    print(f"\n  [{r.id}] {r.type} · {r.priority} · conf={r.confidence}")
    print(f"    Statement: {r.statement}")
    print(f"    Source: '{r.source_utterance[:80]}...'")
    print(f"    Speaker: {r.source_speaker}")

## Step 6 — Run the pipeline on all chunks

In [ ]:
all_requirements = []
for i, chunk in enumerate(chunks, start=1):
    reqs = extract_from_chunk(chunk, i)
    print(f"Chunk {i}: {len(reqs)} requirement(s)")
    all_requirements.extend(reqs)

print(f"\n═══════════════════════════════════════")
print(f"TOTAL EXTRACTED: {len(all_requirements)} requirements")
print(f"═══════════════════════════════════════")

# Summary by type
from collections import Counter
type_counts = Counter(r.type for r in all_requirements)
for t, n in type_counts.items():
    print(f"  {t}: {n}")

## Step 7 — Verify traceability

The critical check: every source_utterance must actually exist in the transcript.

In [ ]:
def verify_traceability(reqs: list[Requirement], transcript: str) -> tuple[list[Requirement], list[Requirement]]:
    verified = []
    unverified = []
    for r in reqs:
        # Fuzzy check: source_utterance should be substantially in transcript
        # (allow minor whitespace/punctuation differences)
        source_normalized = " ".join(r.source_utterance.split())
        transcript_normalized = " ".join(transcript.split())
        if source_normalized in transcript_normalized:
            verified.append(r)
        else:
            # Try substring match on the middle 60% of source
            snippet = source_normalized[len(source_normalized)//5 : 4*len(source_normalized)//5]
            if snippet in transcript_normalized:
                verified.append(r)
            else:
                unverified.append(r)
    return verified, unverified


verified, unverified = verify_traceability(all_requirements, TRANSCRIPT)
print(f"✓ Traceability verified: {len(verified)}/{len(all_requirements)}")
if unverified:
    print(f"\n⚠  Unverified (source not in transcript):")
    for r in unverified:
        print(f"  [{r.id}] source: '{r.source_utterance[:80]}'")
    print("  → These may be hallucinated. Review or drop.")
else:
    print("\n✓ All source_utterances trace to the transcript.")

## Step 8 — Handle withdrawals

Marco revised his 50% → 40% target. The agent should have flagged this.

In [ ]:
withdrawals = [r for r in verified if r.is_withdrawal]
print(f"Withdrawals detected: {len(withdrawals)}")
for w in withdrawals:
    print(f"  [{w.id}] {w.statement}")
    print(f"    Source: {w.source_utterance[:100]}")

# In production: flag any earlier requirement about "50% reduction" as superseded
# For lab: just show which requirements now need review
if withdrawals:
    print("\n→ In production: cross-reference these with earlier requirements. Mark superseded ones as withdrawn.")

## Step 9 — Simple deduplication

Similar statements → merge. Uses word-overlap (production would use embeddings).

In [ ]:
def word_overlap(a: str, b: str) -> float:
    wa = set(a.lower().split())
    wb = set(b.lower().split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)


def dedup_requirements(reqs: list[Requirement], threshold: float = 0.6) -> list[Requirement]:
    kept = []
    for r in reqs:
        is_dup = False
        for k in kept:
            if r.type == k.type and word_overlap(r.statement, k.statement) > threshold:
                is_dup = True
                print(f"  Dedup: [{r.id}] merges into [{k.id}] (overlap > {threshold})")
                break
        if not is_dup:
            kept.append(r)
    return kept


deduped = dedup_requirements(verified)
print(f"\nAfter dedup: {len(deduped)}/{len(verified)} requirements retained")

## Step 10 — Final extraction summary

In [ ]:
print("═" * 60)
print("FINAL REQUIREMENTS SET")
print("═" * 60)
for r in sorted(deduped, key=lambda x: (x.type, x.id)):
    marker = "[WITHDRAWN]" if r.is_withdrawal else ""
    print(f"\n[{r.id}] {r.type.upper()} · {r.priority} · conf={r.confidence} {marker}")
    print(f"  {r.statement}")
    print(f"  Said by: {r.source_speaker}")


# Save for Lab 2
import json
requirements_json = [r.model_dump() for r in deduped]
Path("/tmp/day6_requirements.json").write_text(json.dumps(requirements_json, indent=2))
print(f"\n✓ Saved {len(deduped)} requirements to /tmp/day6_requirements.json for Lab 2")

## What you learned

1. **Chunk by speaker turn** with 1-turn overlap — preserves context, catches split requirements
2. **Pydantic schema is the contract** — no free text, everything typed and validated
3. **`llm.with_structured_output()`** forces schema compliance via native tool-calling
4. **Traceability check** — verify every `source_utterance` exists in transcript, catch hallucinations
5. **Withdrawal detection** — the model can flag "actually, scratch that" as `is_withdrawal=True`
6. **Simple dedup** on word overlap — production would use cosine similarity on embeddings

## Common issues

- Extraction returns nothing → chunks too small, no context
- Source_utterance paraphrased → tighten prompt: 'Copy verbatim'
- Dedup too aggressive → raise threshold to 0.75

## Next

Open `lab2_brd_signoff.ipynb` for document generation + HITL sign-off.
